In [1]:
import caskade as ck
import torch
import numpy as np
from torch.autograd.functional import hessian
import matplotlib.pyplot as plt
import matplotlib.animation as animation
from matplotlib.patches import Ellipse
from matplotlib import colormaps
from IPython.display import HTML
from scipy.optimize import minimize
import emcee

import matplotlib
import matplotlib.pyplot as plt
import numpy as np
import torch
from IPython.display import HTML
from matplotlib import animation
from matplotlib.patches import Circle

import caustics
from caustics.constants import arcsec_to_rad

import numpy as np
import matplotlib.pyplot as plt
import matplotlib.animation as animation
from matplotlib.patches import Ellipse
from matplotlib import colormaps
from IPython.display import HTML

from torch.nn.functional import conv2d, avg_pool2d
import torch
from torch import pi

import caskade as ck
import caustics
from caustics import Module, forward, Param


from typing import Optional, Union, Annotated
from torch import Tensor
from caskade import forward, Param
# from .base import Source, NameType
# from . import func

import copy
import h5py
import os
import scipy
import sncosmo
from PIL import Image
import galsim

from astropy.cosmology import WMAP9 as cosmo
from astropy.table import Table
from astropy.io import fits
from astropy import wcs

import matplotlib as mpl
mpl.rcParams['mathtext.fontset'] = 'stix'
mpl.rcParams['font.family'] = 'serif'

In [21]:
class Gaussian(ck.Module):
    def __init__(self, name, x0=None, y0=None, q=None, phi=None, sigma=None, flux=None):
        super().__init__(name)
        self.x0 = ck.Param("x0", x0)  # position
        self.y0 = ck.Param("y0", y0)
        self.q = ck.Param("q", q)  # axis ratio
        self.phi = ck.Param("phi", phi)  # orientation
        self.sigma = ck.Param("sigma", sigma)  # width
        self.flux = ck.Param("flux", flux)  # total light

    @ck.forward
    def _r(self, x, y, x0=None, y0=None, q=None, phi=None):
        x, y = x - x0, y - y0
        s, c = torch.sin(phi), torch.cos(phi)
        x, y = c * x - s * y, s * x + c * y
        return (x**2 + (y * q) ** 2).sqrt()

    @ck.forward
    def brightness(self, x, y, sigma=None, flux=None):
        return flux * (-self._r(x, y) ** 2 / sigma**2).exp() / (2 * torch.pi * sigma**2).sqrt()


class Gaussian1D(ck.Module):
    def __init__(self, name, t0=None, sigma=None, peak_flux=None):
        super().__init__(name)
        self.t0 = ck.Param("t0", t0)  # position
        self.sigma = ck.Param("sigma", sigma)  # width
        self.peak_flux = ck.Param("peak_flux", peak_flux)  # intensity

    @ck.forward
    def flux(self, t, peak_flux, t0, sigma):
        return peak_flux * (-(((t + t0) / sigma) ** 2)).exp()


class Combined(ck.Module):
    def __init__(self, name, x, y, models):
        super().__init__(name)
        self.x = x
        self.y = y
        self.models = models

    @ck.forward
    def __call__(self):
        return sum(model.brightness(self.x, self.y) for model in self.models)


class Noiser(ck.Module):
    def __init__(self, name, model, read_noise=0.1, exp_time=100):
        super().__init__(name)
        self.model = model
        self.read_noise = read_noise
        self.exp_time = exp_time

    @ck.forward
    def __call__(self):
        img = self.model()
        read_noise = torch.randn_like(img) * self.read_noise
        poisson_noise = torch.randn_like(img) * (img * self.exp_time).sqrt() / self.exp_time
        return img + read_noise + poisson_noise

In [16]:
Nobs = 5
imgsize = 50
sigma_read = 0.1
exp_time = 25
imgx, imgy = torch.meshgrid(
    torch.linspace(-1, 1, imgsize), torch.linspace(-1, 1, imgsize), indexing="ij"
)
SN = Gaussian("SN", x0=-0.35, y0=-0.2, q=1.0, phi=0.0, sigma=0.05)
SN_lightcurve = Gaussian1D("lightcurve", t0=-3.0, sigma=2.0, peak_flux=0.25)
time = ck.Param("time")
SN.flux = lambda p: p.lightcurve.flux(p.time.value)
SN.flux.link((SN_lightcurve, time))
Galaxy = Gaussian("Galaxy", x0=0.2, y0=0.2, q=0.6, phi=0.5, sigma=0.3, flux=1.0)
sim = Combined("sim", imgx, imgy, [SN, Galaxy])

In [17]:
B = 64
fig, ax = plt.subplots()
times = torch.linspace(0, 6, B)
img = ax.imshow(sim([times[0]]), origin="lower", vmin=0, vmax=1.5)
ax.set_title("Brightness at time 0")


def update(i):
    img.set_data(sim([times[i]]).detach().numpy())
    ax.set_title(f"Brightness at time {times[i]:.2f}")
    return img


ani = animation.FuncAnimation(fig, update, frames=B, interval=60)

plt.close()

# Or display the animation inline
HTML(ani.to_jshtml())

# Time Delays

In [27]:
# setup stuff to define a grid of points to make images

n_pix = 256
res = 0.015
fov = n_pix * res
upsample_factor = 2
z_l = torch.tensor(0.5, dtype=torch.float32)
z_s = torch.tensor(1.0, dtype=torch.float32)

F = 100
source = caustics.Sersic(x0=0.1, y0=0.1, q=1.0, phi=0.0, n=2, Re=0.05)
T = caustics.Param("T")  # time variable
source.Ie = (
    lambda p: torch.sin(2 * 2 * np.pi * (p["T"].value % (2 * np.pi))) ** 2
)  # periodic brightness
source.Ie.link(T)
cosmology = caustics.FlatLambdaCDM()
lens = caustics.SIE(
    cosmology=cosmology,
    x0=0.0,
    y0=0.0,
    q=0.5,
    phi=0,
    Rein=1.0,
    z_l=z_l,
    z_s=z_s,
)
t = torch.linspace(0, 1, F)
theta_x, theta_y = caustics.utils.meshgrid(res, n_pix)
source_images = torch.vmap(lambda t: source.brightness(theta_x, theta_y, [t]))(t)
# Time delay fields
td = lens.time_delay(theta_x, theta_y)
print(f"Delays: {td.min().item()} to {td.max().item()}")
# Raytrace including time delay
beta_x, beta_y = lens.raytrace(theta_x, theta_y)
lensed_images = torch.vmap(lambda t: source.brightness(beta_x, beta_y, [t + td]))(
    t,
)

Delays: -273.3621520996094 to 68.90985870361328


In [26]:
# Create animation
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(8, 4))

# Display the first frame of the image in the first subplot
im1 = ax1.imshow(
    source_images[0].numpy(),
    cmap="grey",
    interpolation="bilinear",
    vmin=0,
    vmax=source_images.max().item(),
)
ax1.set_title("unlensed")
ax1.axis("off")
im2 = ax2.imshow(
    lensed_images[0].numpy(),
    cmap="grey",
    interpolation="bilinear",
    vmin=0,
    vmax=2,
    extent=[-fov / 2, fov / 2, -fov / 2, fov / 2],
)
ax2.set_title("lensed")
ax2.axis("off")


def update(frame):
    """Update function for the animation."""
    # Update the image in the first subplot
    im1.set_array(source_images[frame].numpy())
    im2.set_array(lensed_images[frame].numpy())

    return im1, im2


ani = animation.FuncAnimation(fig, update, frames=lensed_images.shape[0], interval=60)

plt.close()
HTML(ani.to_jshtml())

# Time delays - our system

In [2]:
# Goldstein 2019 file
f = h5py.File('lsst-altsched-1a-lowz.h5','r')
simitems = np.asarray(f['system']['block0_items'][()],dtype=str)
simdataall = f['system']['block0_values'][()]

print(simitems.shape, simdataall.shape)
print(simitems[:32])
ind = 96679
simdata=simdataall[ind]

z_lens = simdata[simitems == "zl"][0]
z_source = simdata[simitems == "zs"][0]
print('z lens',z_lens,'z source',z_source)

sn_x = simdata[simitems == "snx"][0]
sn_y = simdata[simitems == "sny"][0]

host_reff = simdata[simitems == "host_reff"][0]      # units? assuming arcsec
host_n = simdata[simitems == "host_n"][0]          
host_x = simdata[simitems == "host_x"][0]            # units? assuming arsec
host_y = simdata[simitems == "host_y"][0]            # units? assuming arcsec
host_theta = simdata[simitems == "host_theta"][0]    # units? assuming radians
host_ellip = simdata[simitems == "host_ellip"][0]
#host_mag = ???? 
sh = galsim.Shear(e=host_ellip, beta=host_theta*galsim.radians)
host_e1 = sh.e1
host_e2 = sh.e2

lens_reff = simdata[simitems == "lensgal_reff"][0]    # units? assuming arcsec
lens_n = simdata[simitems == "lensgal_n"][0]
lens_x = simdata[simitems == "lensgal_x"][0]          # units? assuming arcsec
lens_y = simdata[simitems == "lensgal_y"][0]          # units? assuming arcsec
theta_E_sim = simdata[simitems == "theta_e"][0]       # units? idl what is going on here
lens_theta = simdata[simitems == "lensgal_theta"][0]  # units? assuming degrees
lens_ellip = simdata[simitems == "lensgal_ellip"][0]
s = galsim.Shear(e=lens_ellip, beta=lens_theta*galsim.degrees)
lens_e1 = s.e1
lens_e2 = s.e2

# Einstein radius calculation done by Charlotte in OG Lenstronomy code - just copied
Dl = cosmo.angular_diameter_distance(simdata[simitems == "zl"][0]).value
Ds = cosmo.angular_diameter_distance(simdata[simitems == "zs"][0]).value
Dls = cosmo.angular_diameter_distance_z1z2(simdata[simitems == "zl"][0],simdata[simitems == "zs"][0]).value
sig = simdata[simitems == "sigma"][0]
theta_E = 4*np.pi*(sig/3e5)**2*Dls/Ds*180/np.pi*60*60    # units = arcsec??

lsst_mag_zero_point_g, lsst_mag_zero_point_r, lsst_mag_zero_point_i = 28.3, 28.13, 27.79

(32,) (233744, 32)
['t0' 'sigma' 'gamma' 'e' 'theta_e' 'theta_gamma' 'zs' 'zl' 'snx' 'sny'
 'MB' 'transient_amplitude' 'ebvhost' 'ra' 'dec' 'ebvmw' 'weight' 'host_n'
 'host_ellip' 'host_reff' 'host_theta' 'host_amplitude' 'host_x' 'host_y'
 'lensgal_n' 'lensgal_ellip' 'lensgal_reff' 'lensgal_theta'
 'lensgal_amplitude' 'lensgal_x' 'lensgal_y' 't_found']
z lens 0.35053917396292883 z source 0.6728093548400655


In [3]:
import numpy as np
import torch
import sncosmo
import caustics

# --- your sncosmo helper functions (same as you wrote) ---
def sncosmo_lightcurve(times, band, zero_point, z_source=0.0):
    model = sncosmo.Model(source="salt2")
    model.set(z=z_source, t0=50, x0=1.0e-5, x1=0.1, c=-0.1)
    fluxes = model.bandflux(band, times, zp=zero_point, zpsys="ab")
    return times, fluxes

def make_caustics_lightcurve_func(t_min, t_max, band, zp, z_source):
    times_np = np.linspace(t_min, t_max, 2000)
    _, fluxes_np = sncosmo_lightcurve(times_np, band, zp, z_source)

    times_tensor  = torch.tensor(times_np, dtype=torch.float32)   # (2000,)
    fluxes_tensor = torch.tensor(fluxes_np, dtype=torch.float32)  # (2000,)

    def lc_func(t):
        return caustics.utils.interp1d(
            times_tensor,
            fluxes_tensor,
            t.reshape(-1),
        ).reshape(t.shape)

    return lc_func

# pick ONE band for this example
sn_lc_func_g = make_caustics_lightcurve_func(
    t_min=0,
    t_max=150,
    band="lsstg",
    zp=lsst_mag_zero_point_g,   # you already have this
    z_source=float(z_source), # make sure this matches your source redshift
)

# --- original caustics example (minimal edits below) ---

n_pix = 256
res = 0.015
F = 100
pixscale = 0.11

z_l = torch.tensor(z_lens, dtype=torch.float32)
z_s = torch.tensor(z_source, dtype=torch.float32)

source = caustics.Sersic(x0=sn_x, y0=sn_y, q=1.0, phi = 0.0, n = 4.0, Re=pixscale / 50, Ie = 1.0)

T = caustics.Param("T")  # time variable

# REPLACE the toy periodic function with sncosmo lc_func
source.Ie = lambda p: sn_lc_func_g(p["T"].value)

# keep the same linking mechanism
source.Ie.link(T)

cosmology = caustics.FlatLambdaCDM()

lens_mass = caustics.SIE(name="sie", cosmology=cosmology, z_s = z_source, z_l = z_lens,
                         x0 = lens_x,y0 = lens_y, Rein = theta_E, 
                         angle_system = "e1_e2", e1 = lens_e1, e2 = lens_e2)

lens_mass.to_static(local_only=False)
source.to_static(local_only=False)  


# IMPORTANT: make t have units consistent with your lightcurve domain.
# Here your lightcurve is 0..150 "days" (as you described).
t = torch.linspace(0, 150, F, dtype=torch.float32)

theta_x, theta_y = caustics.utils.meshgrid(res, n_pix)

# unlensed source movie
source_images = torch.vmap(lambda tt: source.brightness(theta_x, theta_y, [tt]))(t)

# time delay field (check units: caustics may return seconds; many examples divide by 365 etc.)
td = lens_mass.time_delay(theta_x, theta_y)
print(f"Delays: {td.min().item()} to {td.max().item()}")

# raytrace and apply time delays
beta_x, beta_y = lens_mass.raytrace(theta_x, theta_y)

lensed_images = torch.vmap(
    lambda tt: source.brightness(beta_x, beta_y, [tt + td])
)(t)


Delays: -79.42149353027344 to 5.825272560119629


In [51]:
# Create animation
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(8, 4))

# Display the first frame of the image in the first subplot
im1 = ax1.imshow(
    source_images[0].numpy(),
    cmap="grey",
    interpolation="bilinear",
    vmin=0,
    vmax=source_images.max().item(),
)
ax1.set_title("unlensed")
ax1.axis("off")
im2 = ax2.imshow(
    lensed_images[0].numpy(),
    cmap="grey",
    interpolation="bilinear",
    vmin=0,
    vmax=2,
    extent=[-fov / 2, fov / 2, -fov / 2, fov / 2],
)
ax2.set_title("lensed")
ax2.axis("off")

time_text = fig.suptitle(f"t = {t[0].item():.1f}")
def update(frame):
    """Update function for the animation."""
    # Update the image in the first subplot
    im1.set_array(source_images[frame].numpy())
    im2.set_array(lensed_images[frame].numpy())
    
    time_text.set_text(f"t = {t[frame].item():.1f}")
    return im1, im2


ani = animation.FuncAnimation(fig, update, frames=lensed_images.shape[0], interval=60)

plt.close()
HTML(ani.to_jshtml())